# Final SGCA Model Comparison and Publication Figures

This notebook records the manuscript comparison workflow for the two final SGCA-family models: Unified SGCA and SGCA Cross-Attention. It evaluates validation, internal test, and development-external cohorts under disease-only and species-conditioned decoding, then exports paper-ready figures.

**Publication release note.** Raw images, per-image prediction files, probability arrays, and figure source tables are intentionally withheld from the GitHub-ready folder at this stage. The final rendered figures are included under `figures/model_comparison/`, and the full source-data package can be restored later for complete regeneration.


## Reproducibility Boundary

The final Unified SGCA checkpoint used for the manuscript is the dog-normal robustness fine-tuned checkpoint. The development-external cohort is used as a robustness cohort, not as a prospective clinical validation set. Species-conditioned decoding uses the known host species supplied with each sample and represents the deployment protocol used for headline reporting.


In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_curve,
)

from sgca.models import (
    CLASS_TO_SPECIES,
    build_dataloaders_from_split_dirs,
    build_evaluation_loader,
    index_species_disease_dataset,
    load_checkpoint,
    seed_everything,
)

warnings.filterwarnings("default")

In [ ]:
ROOT = Path.cwd()
SPLIT_ROOT = ROOT / "data" / "training_data_deduped_splits" / "seed42"
EXTERNAL_ROOT = ROOT / "data" / "development_external_2026_05_02"
OUT_DIR = ROOT / "figures" / "model_comparison"
SOURCE_DIR = OUT_DIR / "source_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "unified": {
        "display_name": "Unified SGCA",
        "checkpoint": ROOT / "results" / "models" / "unified_sgca" / "unified_sgca_best.pt",
        "history": ROOT / "results" / "models" / "unified_sgca" / "training_history.csv",
        "color": "#0072B2",  # Okabe-Ito blue
    },
    "crossattention": {
        "display_name": "SGCA Cross-Attention",
        "checkpoint": ROOT / "results" / "models" / "sgca_cross_attention" / "sgca_cross_attention_best.pt",
        "history": ROOT / "results" / "models" / "sgca_cross_attention" / "training_history.csv",
        "color": "#D55E00",  # Okabe-Ito vermillion
    },
}

SPLITS = ["val", "internal_test", "development_external"]
MODES = ["raw", "known_species"]
BOOTSTRAP_RESAMPLES = 1000
CURVE_BOOTSTRAP_RESAMPLES = 500
SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 0

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Output:", OUT_DIR)
for k, info in MODELS.items():
    print(k, "checkpoint exists:", info["checkpoint"].exists())

In [ ]:
ARIAL_FONT_PATHS = [
    Path("C:/Windows/Fonts/arial.ttf"),
    Path("C:/Windows/Fonts/arialbd.ttf"),
    Path("/mnt/c/Windows/Fonts/arial.ttf"),
    Path("/mnt/c/Windows/Fonts/arialbd.ttf"),
]
for fp in ARIAL_FONT_PATHS:
    if fp.exists():
        try:
            font_manager.fontManager.addfont(str(fp))
        except Exception:
            pass

sns.set_style("white")
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    "font.family": "Arial",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
# Paper-quality display strings for every code-side identifier.
DISEASE_DISPLAY = {
    "Cat_normal": "Cat \u2014 Normal skin",
    "Dog_normal": "Dog \u2014 Normal skin",
    "Normal Skin": "Cattle \u2014 Normal skin",
    "Ringworm in Cat": "Ringworm in cat",
    "Ringworm in Dog": "Ringworm in dog",
    "Ringworm(cow)": "Ringworm in cattle",
    "Skin Allergy in Cat": "Skin allergy in cat",
    "Skin Allergy in Dog": "Skin allergy in dog",
    "Ear Mites in Cat": "Ear mites in cat",
    "Eye Infection in Cat": "Eye infection in cat",
    "Eye Infection in Dog": "Eye infection in dog",
    "Dermatitis in Dog": "Dermatitis in dog",
    "Fungal Infection in Dog": "Fungal infection in dog",
    "Hot Spots in Dog": "Hot spots in dog",
    "Mange in Dog": "Mange in dog",
    "Tick Infestation in Dog": "Tick infestation in dog",
    "Foot and Mouth disease": "Foot-and-mouth disease",
    "Lumpy Skin": "Lumpy skin disease",
    "papiloma": "Papilloma",
    "scabies cat": "Scabies in cat",
    "scabies cattle": "Scabies in cattle",
}

SPLIT_DISPLAY = {
    "val": "Validation",
    "internal_test": "Internal test",
    "development_external": "External validation",
}

MODE_DISPLAY = {
    "raw": "Disease-only decoding",
    "known_species": "Species-conditioned decoding",
}

SPECIES_DISPLAY = {"Cat": "Cat", "Cattles": "Cattle", "Dog": "Dog"}

DISEASE_ORDER = [
    "Cat_normal", "Ringworm in Cat", "Skin Allergy in Cat", "scabies cat",
    "Ear Mites in Cat", "Eye Infection in Cat",
    "Dog_normal", "Dermatitis in Dog", "Fungal Infection in Dog", "Hot Spots in Dog",
    "Mange in Dog", "Ringworm in Dog", "Skin Allergy in Dog",
    "Tick Infestation in Dog", "Eye Infection in Dog",
    "Normal Skin", "Foot and Mouth disease", "Lumpy Skin",
    "Ringworm(cow)", "scabies cattle", "papiloma",
]


def disp_disease(name):
    return DISEASE_DISPLAY.get(name, str(name).replace("_", " "))


def disp_species(name):
    return SPECIES_DISPLAY.get(name, name)


def order_diseases(names):
    rank = {n: i for i, n in enumerate(DISEASE_ORDER)}
    return sorted(set(names), key=lambda x: (rank.get(x, 999), x))


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")


def save_figure(fig, name):
    png = OUT_DIR / f"{name}.png"
    pdf = OUT_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("saved", png.name)


def percent_axis(ax, axis="y"):
    fmt = mpl.ticker.PercentFormatter(xmax=1.0)
    if axis == "y":
        ax.yaxis.set_major_formatter(fmt)
    else:
        ax.xaxis.set_major_formatter(fmt)


def softmax_np(x, axis=1):
    x = x - x.max(axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=axis, keepdims=True)

## Load data

Loaders for clean validation, internal test, and the May-2 development external cohort. Image size is read from the checkpoint so both models are evaluated at the same resolution.

In [ ]:
first_ckpt_path = next(iter(MODELS.values()))["checkpoint"]
_, first_ckpt = load_checkpoint(first_ckpt_path, pretrained=False, device=DEVICE)
species_names = list(first_ckpt["species_names"])
disease_names = list(first_ckpt["disease_names"])
img_size = int(first_ckpt["img_size"])
print("Species:", species_names)
print("Diseases:", len(disease_names))
print("Image size:", img_size)

LOADERS = {}
data = build_dataloaders_from_split_dirs(
    SPLIT_ROOT / "train", SPLIT_ROOT / "val", SPLIT_ROOT / "test",
    img_size=img_size, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    augmentation_policy="standard",
)
LOADERS["val"] = data.val_dl
LOADERS["internal_test"] = data.test_dl

external_frame = index_species_disease_dataset(EXTERNAL_ROOT)
external_loader, external_df = build_evaluation_loader(
    external_frame, img_size=img_size, species_names=species_names,
    disease_names=disease_names, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
)
LOADERS["development_external"] = external_loader

EXTERNAL_DISEASES = sorted(external_df["disease_name"].unique().tolist())
print("Validation:", len(data.val_df))
print("Internal test:", len(data.test_df))
print("External validation:", len(external_df), f"images covering {len(EXTERNAL_DISEASES)} of {len(disease_names)} disease classes")

## Inference

Two decoding modes are computed per image:

- **Disease-only decoding** — argmax over all 21 disease logits.
- **Species-conditioned decoding** — disease logits restricted to the diseases valid for the **true (metadata) species**. This is the deployment protocol.

In [ ]:
def valid_disease_indices_by_species(species_names, disease_names):
    out = {}
    for sp_idx, sp_name in enumerate(species_names):
        valid = [i for i, d in enumerate(disease_names) if CLASS_TO_SPECIES.get(d) == sp_name]
        if not valid:
            raise ValueError(f"No diseases mapped to {sp_name!r}")
        out[sp_idx] = np.array(valid, dtype=int)
    return out


VALID_BY_SPECIES = valid_disease_indices_by_species(species_names, disease_names)


@torch.no_grad()
def collect_predictions(model_key, model_info, split_name, loader):
    model, ckpt = load_checkpoint(model_info["checkpoint"], pretrained=False, device=DEVICE)
    model.eval()
    rows = []
    y_species, pred_species, y_disease = [], [], []
    pred_raw, pred_known = [], []
    species_score_rows, raw_score_rows, known_score_rows = [], [], []
    for images, species_labels, disease_labels, paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        species_logits, disease_logits = model(images)
        species_logits = species_logits.detach().cpu().numpy()
        disease_logits = disease_logits.detach().cpu().numpy()
        species_probs = softmax_np(species_logits, axis=1)
        raw_probs = softmax_np(disease_logits, axis=1)
        true_species = species_labels.numpy()
        true_disease = disease_labels.numpy()
        species_pred = species_probs.argmax(axis=1)
        raw_pred = raw_probs.argmax(axis=1)
        known_probs = np.zeros_like(raw_probs)
        known_pred = np.zeros_like(raw_pred)
        for i, sp_idx in enumerate(true_species):
            valid = VALID_BY_SPECIES[int(sp_idx)]
            local = softmax_np(disease_logits[i, valid][None, :], axis=1).ravel()
            known_probs[i, valid] = local
            known_pred[i] = int(valid[np.argmax(local)])
        y_species.append(true_species)
        pred_species.append(species_pred)
        y_disease.append(true_disease)
        pred_raw.append(raw_pred)
        pred_known.append(known_pred)
        species_score_rows.append(species_probs)
        raw_score_rows.append(raw_probs)
        known_score_rows.append(known_probs)
        for i, p in enumerate(paths):
            rows.append({
                "path": p,
                "true_species": species_names[int(true_species[i])],
                "pred_species": species_names[int(species_pred[i])],
                "true_disease": disease_names[int(true_disease[i])],
                "pred_disease_raw": disease_names[int(raw_pred[i])],
                "pred_disease_known": disease_names[int(known_pred[i])],
                "raw_confidence": float(raw_probs[i, raw_pred[i]]),
                "known_confidence": float(known_probs[i, known_pred[i]]),
            })
    arrays = {
        "y_species": np.concatenate(y_species),
        "pred_species": np.concatenate(pred_species),
        "y_disease": np.concatenate(y_disease),
        "pred_raw": np.concatenate(pred_raw),
        "pred_known": np.concatenate(pred_known),
        "species_scores": np.concatenate(species_score_rows),
        "raw_scores": np.concatenate(raw_score_rows),
        "known_scores": np.concatenate(known_score_rows),
    }
    return pd.DataFrame(rows), arrays


# Set RUN_INFERENCE=False to skip inference and reload cached predictions
# from source_data/. Useful for figure-only iterations after the first full run.
RUN_INFERENCE = True

PREDICTIONS = {}
for mk, mi in MODELS.items():
    PREDICTIONS[mk] = {}
    for sn, ld in LOADERS.items():
        df_path = SOURCE_DIR / f"predictions_{mk}_{sn}.csv"
        npz_path = SOURCE_DIR / f"arrays_{mk}_{sn}.npz"
        if not RUN_INFERENCE and df_path.exists() and npz_path.exists():
            print(f"Loading cached: {mi['display_name']} / {SPLIT_DISPLAY[sn]}")
            df = pd.read_csv(df_path)
            npz = np.load(npz_path)
            arr = {k: npz[k] for k in npz.files}
            PREDICTIONS[mk][sn] = {"df": df, "arrays": arr}
        else:
            print(f"Predicting {mi['display_name']} on {SPLIT_DISPLAY[sn]} ...")
            df, arr = collect_predictions(mk, mi, sn, ld)
            PREDICTIONS[mk][sn] = {"df": df, "arrays": arr}
            df.to_csv(df_path, index=False)
            np.savez_compressed(npz_path, **arr)
print("Done.")

## Metric summary table (with bootstrap 95% CIs)

In [ ]:
def bootstrap_metric(y, pred, fn, n=BOOTSTRAP_RESAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    nrec = len(y)
    vals = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, nrec, nrec)
        vals[i] = fn(y[idx], pred[idx])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def macro_f1_present(y, pred):
    labels = sorted(np.unique(y).tolist())
    return f1_score(y, pred, labels=labels, average="macro", zero_division=0)


def topk_acc(y_true, scores, k=3):
    top = np.argsort(scores, axis=1)[:, ::-1][:, :k]
    return float(np.mean([y_true[i] in top[i] for i in range(len(y_true))]))


def bin_one_vs_rest(y, scores, labels):
    y_bin = np.zeros((len(y), len(labels)), dtype=int)
    for j, lab in enumerate(labels):
        y_bin[:, j] = (y == lab).astype(int)
    return y_bin, scores[:, labels]


def aggregate_aucs(y, scores):
    labels = sorted(np.unique(y).tolist())
    y_bin, score = bin_one_vs_rest(y, scores, labels)
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), score.ravel())
    roc_micro = auc(fpr_micro, tpr_micro)
    pr_micro = average_precision_score(y_bin, score, average="micro")
    aucs, aps = [], []
    for j in range(len(labels)):
        s = y_bin[:, j].sum()
        if s == 0 or s == len(y_bin):
            continue
        f, t, _ = roc_curve(y_bin[:, j], score[:, j])
        aucs.append(auc(f, t))
        aps.append(average_precision_score(y_bin[:, j], score[:, j]))
    return {
        "roc_micro": float(roc_micro),
        "pr_micro": float(pr_micro),
        "roc_macro": float(np.mean(aucs)) if aucs else float("nan"),
        "pr_macro": float(np.mean(aps)) if aps else float("nan"),
    }


def bootstrap_aucs(y, scores, n=BOOTSTRAP_RESAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    n_rec = len(y)
    roc_macro = np.full(n, np.nan)
    pr_macro = np.full(n, np.nan)
    for i in range(n):
        idx = rng.integers(0, n_rec, n_rec)
        try:
            r = aggregate_aucs(y[idx], scores[idx])
            roc_macro[i] = r["roc_macro"]
            pr_macro[i] = r["pr_macro"]
        except Exception:
            pass
    return (
        float(np.nanpercentile(roc_macro, 2.5)),
        float(np.nanpercentile(roc_macro, 97.5)),
        float(np.nanpercentile(pr_macro, 2.5)),
        float(np.nanpercentile(pr_macro, 97.5)),
    )


summary_rows = []
for mk in MODELS:
    for sn in SPLITS:
        arr = PREDICTIONS[mk][sn]["arrays"]
        for mode in MODES:
            pred = arr["pred_raw"] if mode == "raw" else arr["pred_known"]
            score = arr["raw_scores"] if mode == "raw" else arr["known_scores"]
            acc = accuracy_score(arr["y_disease"], pred)
            f1m = macro_f1_present(arr["y_disease"], pred)
            acc_lo, acc_hi = bootstrap_metric(arr["y_disease"], pred, accuracy_score)
            f1_lo, f1_hi = bootstrap_metric(arr["y_disease"], pred, macro_f1_present)
            aucs = aggregate_aucs(arr["y_disease"], score)
            # Bootstrap AUC/AP CIs only for the deployment mode (known_species);
            # the headline bars use that mode and the extra resampling is expensive.
            if mode == "known_species":
                print(f"  bootstrap AUC CIs: {MODELS[mk]['display_name']} / {SPLIT_DISPLAY[sn]} ...")
                roc_lo, roc_hi, pr_lo, pr_hi = bootstrap_aucs(arr["y_disease"], score)
            else:
                roc_lo = roc_hi = pr_lo = pr_hi = float("nan")
            sp_acc = accuracy_score(arr["y_species"], arr["pred_species"])
            top3 = topk_acc(arr["y_disease"], score, 3)
            summary_rows.append({
                "model_key": mk,
                "model_name": MODELS[mk]["display_name"],
                "split": sn,
                "split_name": SPLIT_DISPLAY[sn],
                "mode": mode,
                "mode_name": MODE_DISPLAY[mode],
                "n": int(len(arr["y_disease"])),
                "species_accuracy": float(sp_acc),
                "disease_accuracy": float(acc),
                "disease_accuracy_ci_low": acc_lo,
                "disease_accuracy_ci_high": acc_hi,
                "macro_f1": float(f1m),
                "macro_f1_ci_low": f1_lo,
                "macro_f1_ci_high": f1_hi,
                "roc_auc_macro": aucs["roc_macro"],
                "roc_auc_macro_ci_low": roc_lo,
                "roc_auc_macro_ci_high": roc_hi,
                "roc_auc_micro": aucs["roc_micro"],
                "pr_auc_macro": aucs["pr_macro"],
                "pr_auc_macro_ci_low": pr_lo,
                "pr_auc_macro_ci_high": pr_hi,
                "pr_auc_micro": aucs["pr_micro"],
                "top3_accuracy": top3,
            })

summary = pd.DataFrame(summary_rows)
summary.to_csv(SOURCE_DIR / "table_metric_summary.csv", index=False)
display_cols = ["model_name", "split_name", "mode_name", "n",
                "disease_accuracy", "macro_f1", "roc_auc_macro", "pr_auc_macro", "top3_accuracy"]
display(summary[display_cols].round(4))

## Architecture schematic

Block-level overview of the Species-Gated Cross-Attention model. Output is a placeholder for redrawing in Canva, but is paper-grade as exported here.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.axis("off")


def box(x, y, w, h, label, fc, ec):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.18",
        fc=fc, ec=ec, lw=1.1,
    ))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9, weight="bold")


def arrow(x0, y0, x1, y1):
    ax.add_patch(FancyArrowPatch(
        (x0, y0), (x1, y1), arrowstyle="->", mutation_scale=12, lw=1.1, color="#374151",
    ))


box(0.2, 1.8, 1.8, 1.2, "Input image\n384 \u00d7 384 \u00d7 3", "#F3F4F6", "#374151")
box(2.4, 1.8, 2.2, 1.2, "EfficientNetV2-S\nbackbone", "#DBEAFE", "#1E40AF")
box(5.0, 3.2, 2.2, 1.1, "Species head\n3 classes", "#DCFCE7", "#166534")
box(5.0, 0.5, 2.2, 1.1, "Disease head\n21 classes", "#FEE2E2", "#B91C1C")
box(7.6, 1.8, 2.6, 1.2, "Species-Gated\nCross-Attention\n(SGCA)", "#FEF3C7", "#B45309")
box(10.4, 1.8, 1.4, 1.2, "Outputs:\nspecies + disease\n+ uncertainty", "#F3F4F6", "#374151")

arrow(2.0, 2.4, 2.4, 2.4)
arrow(4.6, 2.7, 5.0, 3.7)
arrow(4.6, 2.1, 5.0, 1.05)
arrow(7.2, 3.7, 7.8, 2.7)
arrow(7.2, 1.05, 7.8, 2.1)
arrow(10.2, 2.4, 10.4, 2.4)

ax.text(6.0, 4.7, "Architecture overview \u2014 species head gates disease decoding via cross-attention",
        ha="center", va="center", fontsize=9.5, weight="bold")
fig.tight_layout()
save_figure(fig, "fig_architecture_schematic")

## Cohort flow

In [ ]:
train_df = data.train_df
val_df = data.val_df
test_df = data.test_df

split_order = ["Train", "Validation", "Internal test", "External validation"]
species_split = pd.DataFrame(index=species_names, columns=split_order, dtype=int)
for label, df_ in zip(split_order, [train_df, val_df, test_df, external_df]):
    counts = df_["species_name"].value_counts().reindex(species_names, fill_value=0)
    species_split[label] = counts.values

fig, ax = plt.subplots(figsize=(6.6, 3.2))
bottom = np.zeros(len(split_order))
sp_colors = {"Cat": "#0072B2", "Cattles": "#009E73", "Dog": "#D55E00"}
for sp in species_names:
    vals = species_split.loc[sp].values.astype(int)
    ax.bar(split_order, vals, bottom=bottom, color=sp_colors[sp],
           edgecolor="white", linewidth=0.8, label=disp_species(sp))
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0:
            ax.text(i, b + v / 2, f"{int(v)}", ha="center", va="center",
                    fontsize=8.5, color="white", weight="bold")
    bottom += vals
for i, total in enumerate(bottom):
    ax.text(i, total + max(bottom) * 0.02, f"n = {int(total)}",
            ha="center", va="bottom", fontsize=9, weight="bold")
ax.set_ylabel("Number of images", weight="bold")
ax.set_title("Cohort flow \u2014 image counts by split and species", weight="bold")
ax.set_ylim(0, max(bottom) * 1.18)
ax.legend(title="Species", frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig_cohort_flow")
species_split.to_csv(SOURCE_DIR / "table_cohort_flow.csv")

## Training curves

Validation curves only \u2014 confirms model selection was driven by internal validation behaviour, not external performance.

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.8))
for mk, mi in MODELS.items():
    hist = pd.read_csv(mi["history"])
    ax.plot(hist["epoch"], hist["val_disease_f1"], color=mi["color"],
            linewidth=1.6, label=mi["display_name"])
    best = hist["val_disease_f1"].idxmax()
    ax.scatter([hist.loc[best, "epoch"]], [hist.loc[best, "val_disease_f1"]],
               color=mi["color"], s=36, edgecolor="black", linewidth=0.6, zorder=5)
ax.set_xlabel("Epoch", weight="bold")
ax.set_ylabel("Validation macro-F1", weight="bold")
ax.set_ylim(0.5, 1.0)
percent_axis(ax)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig_training_macroF1_curve")

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.8))
for mk, mi in MODELS.items():
    hist = pd.read_csv(mi["history"])
    ax.plot(hist["epoch"], hist["val_disease_acc"], color=mi["color"],
            linewidth=1.6, label=mi["display_name"])
    best = hist["val_disease_acc"].idxmax()
    ax.scatter([hist.loc[best, "epoch"]], [hist.loc[best, "val_disease_acc"]],
               color=mi["color"], s=36, edgecolor="black", linewidth=0.6, zorder=5)
ax.set_xlabel("Epoch", weight="bold")
ax.set_ylabel("Validation accuracy", weight="bold")
ax.set_ylim(0.5, 1.0)
percent_axis(ax)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig_training_accuracy_curve")

## Confusion matrices (row-normalised, %)

Six files: 2 models \u00d7 3 splits. External cohort uses only the 10 disease classes present in the cohort. Cell font size is enlarged so values fully cover each cell.

In [ ]:
for mk, mi in MODELS.items():
    for sn in SPLITS:
        arr = PREDICTIONS[mk][sn]["arrays"]
        df = PREDICTIONS[mk][sn]["df"]
        if sn == "development_external":
            present = order_diseases(df["true_disease"].unique())
            figsize = (7.5, 7.5)
            annot_size = 16
            tick_size = 11
        else:
            present = order_diseases(disease_names)
            figsize = (12.0, 12.0)
            annot_size = 12
            tick_size = 10
        label_idx = [disease_names.index(d) for d in present]
        text_labels = [disp_disease(d) for d in present]
        cm = confusion_matrix(arr["y_disease"], arr["pred_known"], labels=label_idx, normalize="true") * 100
        fig, ax = plt.subplots(figsize=figsize)
        sns.heatmap(
            cm, ax=ax, cmap="Blues", vmin=0, vmax=100,
            annot=True, fmt=".0f", cbar=True, square=True,
            linewidths=0.5, linecolor="white",
            xticklabels=text_labels, yticklabels=text_labels,
            annot_kws={"size": annot_size, "weight": "bold", "family": "Arial"},
            cbar_kws={"shrink": 0.7, "pad": 0.02},
        )
        ax.set_title(f"{mi['display_name']} \u2014 {SPLIT_DISPLAY[sn]} (species-conditioned, %)",
                     weight="bold", fontsize=12, pad=10)
        ax.set_xlabel("Predicted disease", weight="bold")
        ax.set_ylabel("True disease", weight="bold")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=tick_size, weight="bold")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=tick_size, weight="bold")
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=9)
        cbar.set_label("Row-normalised %", fontsize=10, weight="bold")
        fig.tight_layout()
        save_figure(fig, f"fig_confusion_{mk}_{sn}")

## Per-disease F1 \u2014 paired forest plot

Three files, one per split. Both models shown on the same axes for direct comparison: dot = point estimate, horizontal line = 95% bootstrap CI. Per-class image counts shown on the right.

In [ ]:
def per_class_f1_with_ci(y_true, pred, classes, n_boot=BOOTSTRAP_RESAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    obs = np.zeros(len(classes))
    boot = np.zeros((n_boot, len(classes)))
    for j, c in enumerate(classes):
        obs[j] = f1_score((y_true == c).astype(int), (pred == c).astype(int), zero_division=0)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yi = y_true[idx]; pi = pred[idx]
        for j, c in enumerate(classes):
            boot[b, j] = f1_score((yi == c).astype(int), (pi == c).astype(int), zero_division=0)
    return obs, np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)


# Markers and a dimming colour for the support-count column
MARKERS = {"unified": "o", "crossattention": "s"}
SUPPORT_TEXT_COLOR = "#4B5563"


def render_paired_forest(metric_label, value_xlim, model_data, present, support, sn, fname_stem,
                         x_is_percent=True, xticks=None):
    text_labels = [disp_disease(d) for d in present]
    h = max(3.5, 0.42 * len(present) + 1.2)
    fig, ax = plt.subplots(figsize=(6.4, h))
    y = np.arange(len(present))
    offset = 0.18
    for sign, (mk, mi) in zip([+offset, -offset], MODELS.items()):
        d = model_data[mk]
        valid = ~np.isnan(d["obs"])
        y_pos = y[valid] + sign
        ax.hlines(y_pos, d["lo"][valid], d["hi"][valid],
                  color=mi["color"], linewidth=1.6, alpha=0.85)
        ax.scatter(d["obs"][valid], y_pos, color=mi["color"], s=42,
                   marker=MARKERS[mk], edgecolor="black", linewidth=0.5,
                   zorder=5, label=mi["display_name"])
    ax.set_yticks(y)
    ax.set_yticklabels(text_labels, fontsize=9, weight="bold")
    ax.set_xlim(*value_xlim)
    if xticks is not None:
        ax.set_xticks(xticks)
    if x_is_percent:
        percent_axis(ax, axis="x")
    ax.set_xlabel(metric_label, weight="bold")
    ax.set_title(f"{SPLIT_DISPLAY[sn]} \u2014 {fname_stem.replace('_', ' ')}",
                 weight="bold", fontsize=11)
    ax.grid(axis="x", color="#E5E7EB", linewidth=0.5)
    ax.invert_yaxis()
    sns.despine(ax=ax)
    # Support count column on the right
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(y)
    ax2.set_yticklabels([f"n = {s}" for s in support], fontsize=8, color=SUPPORT_TEXT_COLOR)
    for spine in ("top", "right", "bottom", "left"):
        ax2.spines[spine].set_visible(False)
    ax2.tick_params(axis="y", length=0)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2,
              frameon=False, fontsize=9)
    fig.tight_layout()
    save_figure(fig, f"fig_{fname_stem}_{sn}")
    plt.close(fig)


for sn in SPLITS:
    if sn == "development_external":
        present = order_diseases(PREDICTIONS["unified"][sn]["df"]["true_disease"].unique())
    else:
        present = order_diseases(disease_names)
    label_idx = np.array([disease_names.index(d) for d in present])
    arr0 = PREDICTIONS[next(iter(MODELS))][sn]["arrays"]
    support = [int((arr0["y_disease"] == idx).sum()) for idx in label_idx]

    model_data = {}
    for mk in MODELS:
        arr = PREDICTIONS[mk][sn]["arrays"]
        obs, lo, hi = per_class_f1_with_ci(arr["y_disease"], arr["pred_known"], label_idx)
        model_data[mk] = {"obs": obs, "lo": lo, "hi": hi}

    render_paired_forest(
        metric_label="Disease F1-score (95% CI)",
        value_xlim=(0, 1.10),
        xticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        model_data=model_data, present=present, support=support, sn=sn,
        fname_stem="per_disease_f1", x_is_percent=True,
    )

    rows = []
    for mk in MODELS:
        d = model_data[mk]
        for j, name in enumerate(present):
            rows.append({
                "split": sn, "model": MODELS[mk]["display_name"], "disease": name,
                "f1": float(d["obs"][j]), "ci_low": float(d["lo"][j]),
                "ci_high": float(d["hi"][j]), "support": support[j],
            })
    pd.DataFrame(rows).to_csv(SOURCE_DIR / f"per_disease_f1_{sn}.csv", index=False)

## Per-class ROC-AUC \u2014 paired forest plot

Three files, one per split. Both models on the same axes; one-vs-rest AUC on species-conditioned scores; 95% bootstrap CIs.

In [ ]:
def per_class_auc_with_ci(y_true, scores, classes, n_boot=BOOTSTRAP_RESAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    obs = np.full(len(classes), np.nan)
    boot = np.full((n_boot, len(classes)), np.nan)
    for j, c in enumerate(classes):
        yb = (y_true == c).astype(int)
        if 0 < yb.sum() < n:
            f, t, _ = roc_curve(yb, scores[:, c])
            obs[j] = auc(f, t)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yi = y_true[idx]; si = scores[idx]
        for j, c in enumerate(classes):
            yb = (yi == c).astype(int)
            if 0 < yb.sum() < n:
                try:
                    f, t, _ = roc_curve(yb, si[:, c])
                    boot[b, j] = auc(f, t)
                except Exception:
                    pass
    return obs, np.nanpercentile(boot, 2.5, axis=0), np.nanpercentile(boot, 97.5, axis=0)


for sn in SPLITS:
    if sn == "development_external":
        present = order_diseases(PREDICTIONS["unified"][sn]["df"]["true_disease"].unique())
    else:
        present = order_diseases(disease_names)
    label_idx = np.array([disease_names.index(d) for d in present])
    arr0 = PREDICTIONS[next(iter(MODELS))][sn]["arrays"]
    support = [int((arr0["y_disease"] == idx).sum()) for idx in label_idx]

    model_data = {}
    for mk in MODELS:
        arr = PREDICTIONS[mk][sn]["arrays"]
        obs, lo, hi = per_class_auc_with_ci(arr["y_disease"], arr["known_scores"], label_idx)
        model_data[mk] = {"obs": obs, "lo": lo, "hi": hi}

    render_paired_forest(
        metric_label="One-vs-rest ROC-AUC (95% CI)",
        value_xlim=(0.5, 1.05),
        xticks=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        model_data=model_data, present=present, support=support, sn=sn,
        fname_stem="per_class_auc", x_is_percent=False,
    )

    rows = []
    for mk in MODELS:
        d = model_data[mk]
        for j, name in enumerate(present):
            rows.append({
                "split": sn, "model": MODELS[mk]["display_name"], "disease": name,
                "roc_auc": None if np.isnan(d["obs"][j]) else float(d["obs"][j]),
                "ci_low": None if np.isnan(d["lo"][j]) else float(d["lo"][j]),
                "ci_high": None if np.isnan(d["hi"][j]) else float(d["hi"][j]),
            })
    pd.DataFrame(rows).to_csv(SOURCE_DIR / f"per_class_auc_{sn}.csv", index=False)

## Aggregate ROC curves (with bootstrap CI band)

Three files \u2014 one per split. Both models overlaid, species-conditioned scores.

In [ ]:
def micro_roc_with_band(y, scores, n_boot=CURVE_BOOTSTRAP_RESAMPLES, seed=SEED):
    labels = sorted(np.unique(y).tolist())
    y_bin, score = bin_one_vs_rest(y, scores, labels)
    fpr_obs, tpr_obs, _ = roc_curve(y_bin.ravel(), score.ravel())
    auc_obs = auc(fpr_obs, tpr_obs)
    grid = np.linspace(0, 1, 200)
    rng = np.random.default_rng(seed)
    n = len(y)
    interp = np.zeros((n_boot, len(grid)))
    aucs = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yb_b, sc_b = bin_one_vs_rest(y[idx], scores[idx], labels)
        fb, tb, _ = roc_curve(yb_b.ravel(), sc_b.ravel())
        interp[b] = np.interp(grid, fb, tb)
        interp[b, 0] = 0.0
        aucs[b] = auc(fb, tb)
    return (
        grid,
        np.interp(grid, fpr_obs, tpr_obs),
        np.percentile(interp, 2.5, axis=0),
        np.percentile(interp, 97.5, axis=0),
        float(auc_obs),
        float(np.percentile(aucs, 2.5)),
        float(np.percentile(aucs, 97.5)),
    )


for sn in SPLITS:
    fig, ax = plt.subplots(figsize=(3.7, 3.2))
    for mk, mi in MODELS.items():
        arr = PREDICTIONS[mk][sn]["arrays"]
        grid, tpr_mean, tpr_lo, tpr_hi, a, lo, hi = micro_roc_with_band(arr["y_disease"], arr["known_scores"])
        ax.fill_between(grid, tpr_lo, tpr_hi, color=mi["color"], alpha=0.18, linewidth=0)
        ax.plot(grid, tpr_mean, color=mi["color"], linewidth=1.6,
                label=f"{mi['display_name']}\nAUC {a:.3f} ({lo:.3f}\u2013{hi:.3f})")
    ax.plot([0, 1], [0, 1], color="#9CA3AF", linestyle=":", linewidth=1.0)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("False positive rate", weight="bold")
    ax.set_ylabel("True positive rate", weight="bold")
    ax.set_title(f"{SPLIT_DISPLAY[sn]} \u2014 micro ROC", weight="bold", fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=7)
    ax.grid(color="#E5E7EB", linewidth=0.5)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, f"fig_roc_{sn}")

## Aggregate Precision-Recall curves (with bootstrap CI band)

Three files \u2014 one per split.

In [ ]:
def micro_pr_with_band(y, scores, n_boot=CURVE_BOOTSTRAP_RESAMPLES, seed=SEED):
    labels = sorted(np.unique(y).tolist())
    y_bin, score = bin_one_vs_rest(y, scores, labels)
    p_obs, r_obs, _ = precision_recall_curve(y_bin.ravel(), score.ravel())
    ap_obs = average_precision_score(y_bin, score, average="micro")
    grid = np.linspace(0, 1, 200)
    rng = np.random.default_rng(seed)
    n = len(y)
    interp = np.zeros((n_boot, len(grid)))
    aps = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yb_b, sc_b = bin_one_vs_rest(y[idx], scores[idx], labels)
        p_b, r_b, _ = precision_recall_curve(yb_b.ravel(), sc_b.ravel())
        order = np.argsort(r_b)
        interp[b] = np.interp(grid, r_b[order], p_b[order])
        aps[b] = average_precision_score(yb_b, sc_b, average="micro")
    order = np.argsort(r_obs)
    return (
        grid,
        np.interp(grid, r_obs[order], p_obs[order]),
        np.percentile(interp, 2.5, axis=0),
        np.percentile(interp, 97.5, axis=0),
        float(ap_obs),
        float(np.percentile(aps, 2.5)),
        float(np.percentile(aps, 97.5)),
    )


for sn in SPLITS:
    fig, ax = plt.subplots(figsize=(3.7, 3.2))
    for mk, mi in MODELS.items():
        arr = PREDICTIONS[mk][sn]["arrays"]
        grid, prec_mean, prec_lo, prec_hi, ap, lo, hi = micro_pr_with_band(arr["y_disease"], arr["known_scores"])
        ax.fill_between(grid, prec_lo, prec_hi, color=mi["color"], alpha=0.18, linewidth=0)
        ax.plot(grid, prec_mean, color=mi["color"], linewidth=1.6,
                label=f"{mi['display_name']}\nAP {ap:.3f} ({lo:.3f}\u2013{hi:.3f})")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Recall", weight="bold")
    ax.set_ylabel("Precision", weight="bold")
    ax.set_title(f"{SPLIT_DISPLAY[sn]} \u2014 micro PR", weight="bold", fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=7)
    ax.grid(color="#E5E7EB", linewidth=0.5)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, f"fig_pr_{sn}")

## Reliability diagrams (with ECE)

Three files \u2014 one per split. Reliability uses the species-conditioned predicted disease confidence.

In [ ]:
def reliability_bins(conf, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ids = np.digitize(conf, bins[1:-1], right=True)
    rows, ece = [], 0.0
    for b in range(n_bins):
        m = ids == b
        center = (bins[b] + bins[b + 1]) / 2
        if m.sum() == 0:
            rows.append({"bin": b, "center": center, "accuracy": np.nan, "confidence": np.nan, "n": 0})
            continue
        acc = correct[m].mean()
        avg = conf[m].mean()
        ece += (m.sum() / len(conf)) * abs(acc - avg)
        rows.append({"bin": b, "center": center, "accuracy": acc, "confidence": avg, "n": int(m.sum())})
    return pd.DataFrame(rows), float(ece)


for sn in SPLITS:
    fig, ax = plt.subplots(figsize=(3.7, 3.2))
    for mk, mi in MODELS.items():
        df = PREDICTIONS[mk][sn]["df"]
        conf = df["known_confidence"].to_numpy()
        correct = (df["true_disease"].to_numpy() == df["pred_disease_known"].to_numpy()).astype(float)
        rel, ece = reliability_bins(conf, correct)
        rel_v = rel.dropna()
        ax.plot(rel_v["confidence"], rel_v["accuracy"], marker="o", linewidth=1.6,
                color=mi["color"], markersize=4,
                label=f"{mi['display_name']}\nECE {ece:.3f}")
    ax.plot([0, 1], [0, 1], color="#9CA3AF", linestyle=":", linewidth=1.0)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    percent_axis(ax)
    percent_axis(ax, axis="x")
    ax.set_xlabel("Mean confidence", weight="bold")
    ax.set_ylabel("Empirical accuracy", weight="bold")
    ax.set_title(f"{SPLIT_DISPLAY[sn]} \u2014 calibration", weight="bold", fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
    ax.grid(color="#E5E7EB", linewidth=0.5)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, f"fig_reliability_{sn}")

## Species-subgroup macro-F1

Three files \u2014 one per split. Species-conditioned decoding, with bootstrap 95% CIs.

In [ ]:
species_rows = []
for mk, mi in MODELS.items():
    for sn in SPLITS:
        df = PREDICTIONS[mk][sn]["df"]
        for sp in species_names:
            sub = df[df["true_species"] == sp]
            if sub.empty:
                continue
            y = sub["true_disease"].to_numpy()
            pr = sub["pred_disease_known"].to_numpy()
            acc = accuracy_score(y, pr)
            f1m = f1_score(y, pr, average="macro", zero_division=0)
            f1_lo, f1_hi = bootstrap_metric(y, pr, lambda a, b: f1_score(a, b, average="macro", zero_division=0))
            species_rows.append({
                "model_key": mk, "model_name": mi["display_name"],
                "split": sn, "split_name": SPLIT_DISPLAY[sn],
                "species": sp, "species_display": disp_species(sp),
                "n": int(len(sub)), "accuracy": float(acc),
                "macro_f1": float(f1m), "f1_ci_low": f1_lo, "f1_ci_high": f1_hi,
            })
species_metrics = pd.DataFrame(species_rows)
species_metrics.to_csv(SOURCE_DIR / "table_species_subgroup.csv", index=False)

for sn in SPLITS:
    sub = species_metrics[species_metrics["split"] == sn]
    species_present = [sp for sp in species_names if sp in sub["species"].unique()]
    fig, ax = plt.subplots(figsize=(4.0, 2.9))
    x = np.arange(len(species_present))
    width = 0.34
    for offset, (mk, mi) in zip([-width / 2, width / 2], MODELS.items()):
        d = sub[sub["model_key"] == mk].set_index("species").reindex(species_present)
        vals = d["macro_f1"].values
        err = np.vstack([vals - d["f1_ci_low"].values, d["f1_ci_high"].values - vals])
        ax.bar(x + offset, vals, width=width, color=mi["color"],
               edgecolor="black", linewidth=0.5,
               yerr=err, error_kw={"ecolor": "#374151", "lw": 0.7, "capsize": 2.5},
               label=mi["display_name"])
        for xi, val in zip(x + offset, vals):
            ax.text(xi, val + 0.025, f"{val * 100:.1f}", ha="center", va="bottom", fontsize=7, weight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([disp_species(s) for s in species_present], weight="bold")
    ax.set_ylim(0, 1.15)
    percent_axis(ax)
    ax.set_ylabel("Macro-F1 (95% CI)", weight="bold")
    ax.set_title(f"{SPLIT_DISPLAY[sn]} \u2014 species subgroup", weight="bold", fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, f"fig_species_subgroup_{sn}")

## Headline metric bars across splits

Four files \u2014 one per metric. Both models, three splits, species-conditioned decoding, bootstrap 95% CIs where applicable.

In [ ]:
HEADLINE_METRICS = [
    ("disease_accuracy", "Accuracy", True),
    ("macro_f1", "Macro-F1", True),
    ("roc_auc_macro", "Macro ROC-AUC", True),
    ("pr_auc_macro", "Macro PR-AUC", True),
]

mode_for_headline = "known_species"
for metric, title, has_ci in HEADLINE_METRICS:
    sub = summary[summary["mode"] == mode_for_headline].copy()
    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    x = np.arange(len(SPLITS))
    width = 0.34
    for offset, (mk, mi) in zip([-width / 2, width / 2], MODELS.items()):
        vals, errs_lo, errs_hi = [], [], []
        for sn in SPLITS:
            row = sub[(sub["model_key"] == mk) & (sub["split"] == sn)].iloc[0]
            vals.append(row[metric])
            if has_ci and f"{metric}_ci_low" in sub.columns:
                errs_lo.append(row[metric] - row[f"{metric}_ci_low"])
                errs_hi.append(row[f"{metric}_ci_high"] - row[metric])
            else:
                errs_lo.append(0)
                errs_hi.append(0)
        err = np.array([errs_lo, errs_hi])
        bars = ax.bar(x + offset, vals, width=width, color=mi["color"],
                      edgecolor="black", linewidth=0.5,
                      yerr=err if has_ci else None,
                      error_kw={"ecolor": "#374151", "lw": 0.7, "capsize": 2.5},
                      label=mi["display_name"])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.022,
                    f"{val * 100:.1f}", ha="center", va="bottom", fontsize=8, weight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLITS], weight="bold")
    ax.set_ylim(0, 1.15)
    percent_axis(ax)
    ax.set_ylabel(title, weight="bold")
    ax.set_title(f"{title} \u2014 species-conditioned", weight="bold", fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
    sns.despine(ax=ax)
    fig.tight_layout()
    save_figure(fig, f"fig_headline_{safe_name(metric)}")

## Top-1 vs Top-3 disease accuracy across splits

In [ ]:
labels, top1, top3, colors = [], [], [], []
for sn in SPLITS:
    for mk, mi in MODELS.items():
        labels.append(f"{mi['display_name']}\n{SPLIT_DISPLAY[sn]}")
        row = summary[(summary["model_key"] == mk) & (summary["split"] == sn) & (summary["mode"] == "known_species")].iloc[0]
        top1.append(row["disease_accuracy"])
        top3.append(row["top3_accuracy"])
        colors.append(mi["color"])

fig, ax = plt.subplots(figsize=(5.6, 3.0))
x = np.arange(len(labels))
width = 0.36
ax.bar(x - width / 2, top1, width=width, color=colors, edgecolor="black", linewidth=0.5, label="Top-1")
bars3 = ax.bar(x + width / 2, top3, width=width, color=colors, edgecolor="black", linewidth=0.5,
               hatch="///", label="Top-3")
for xi, v in zip(x - width / 2, top1):
    ax.text(xi, v + 0.015, f"{v * 100:.0f}", ha="center", va="bottom", fontsize=7, weight="bold")
for xi, v in zip(x + width / 2, top3):
    ax.text(xi, v + 0.015, f"{v * 100:.0f}", ha="center", va="bottom", fontsize=7, weight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8, weight="bold")
ax.set_ylim(0, 1.15)
percent_axis(ax)
ax.set_ylabel("Disease classification accuracy", weight="bold")
ax.set_title("Top-1 vs Top-3 accuracy \u2014 species-conditioned", weight="bold", fontsize=11)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig_top1_vs_top3")

## Expected Calibration Error (ECE) across splits

In [ ]:
ece_rows = []
for mk, mi in MODELS.items():
    for sn in SPLITS:
        df = PREDICTIONS[mk][sn]["df"]
        conf = df["known_confidence"].to_numpy()
        correct = (df["true_disease"].to_numpy() == df["pred_disease_known"].to_numpy()).astype(float)
        _, ece = reliability_bins(conf, correct)
        ece_rows.append({"model_key": mk, "model_name": mi["display_name"],
                         "split": sn, "split_name": SPLIT_DISPLAY[sn], "ece": float(ece)})
ece_df = pd.DataFrame(ece_rows)
ece_df.to_csv(SOURCE_DIR / "table_ece.csv", index=False)

fig, ax = plt.subplots(figsize=(4.6, 3.0))
x = np.arange(len(SPLITS))
width = 0.34
for offset, (mk, mi) in zip([-width / 2, width / 2], MODELS.items()):
    vals = [ece_df[(ece_df["model_key"] == mk) & (ece_df["split"] == sn)]["ece"].iloc[0] for sn in SPLITS]
    bars = ax.bar(x + offset, vals, width=width, color=mi["color"],
                  edgecolor="black", linewidth=0.5, label=mi["display_name"])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.004,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8, weight="bold")
ax.set_xticks(x)
ax.set_xticklabels([SPLIT_DISPLAY[s] for s in SPLITS], weight="bold")
ax.set_ylim(0, max(ece_df["ece"]) * 1.5 + 0.005)
ax.set_ylabel("Expected Calibration Error", weight="bold")
ax.set_title("Calibration error across splits \u2014 species-conditioned", weight="bold", fontsize=11)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=8)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, "fig_ece_comparison")

## Output check

In [ ]:
print("Output:", OUT_DIR)
pngs = sorted(OUT_DIR.glob("*.png"))
pdfs = sorted(OUT_DIR.glob("*.pdf"))
csvs = sorted(SOURCE_DIR.glob("*.csv"))
print(f"\nPNG files ({len(pngs)}):")
for p in pngs:
    print(" -", p.name)
print(f"\nPDF files: {len(pdfs)}")
print(f"Source tables: {len(csvs)}")